# 02 - Etapa B: Features

Implementa las **Tasks 1–4** del preprocesamiento de Ferrari et al. (2019):

| Task | Descripción | Output |
|------|-------------|--------|
| 1+2  | Segmentación de pasos (recorte de bordes) | xyz recortado, N pasos, T |
| 3    | 27 ángulos × 3 planos = **81 ángulos/frame** | (n_frames, 81) |
| 4a   | FFT → **1620 features** para MLP | (1620,) por trial |
| 4b   | Ventanas **75×81** con stride 15 para LSTM | (n_seq, 75, 81) por trial |

**Antes de correr:** asegurarse de que `BASE` apunta a `data/diplegia/` con los `.npy` descargados.

In [ ]:
import sys, os
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.data import (
    load_dataset, masked_coords, interpolate_short_gaps,
    analyze_gaps, MARKERS, MARKER_IDX,
)
from src.features import (
    detect_steps, segment_trial,
    compute_projected_angles,
    compute_mlp_features,
    compute_lstm_sequences,
    process_trial, build_dataset,
    PLANES, FS, SUBSAMPLE_FACTOR,
    N_COEFF, SEQ_LEN, SEQ_STEP, MAX_SEQ,
)

BASE    = '../data/diplegia'   # <- cambiar al path real del dataset
OUT_DIR = '../data/features'
os.makedirs(OUT_DIR, exist_ok=True)
MAX_GAP = 5   # igual que en EDA (huecos interiores <= 5 frames se interpolan)

print(f'FPS tras subsampling: {FS} (factor ×{SUBSAMPLE_FACTOR})')
print(f'Planos: {[p for p,_ in PLANES]}')
print(f'Coeficientes FFT: {N_COEFF} → features MLP = {N_COEFF}×81 = {N_COEFF*81}')
print(f'Ventanas LSTM: {SEQ_LEN} frames, stride {SEQ_STEP}, máx {MAX_SEQ}/trial')

## 1. Carga y limpieza (repetir EDA)

In [ ]:
trials_raw = load_dataset(BASE)
print(f'Trials crudos: {len(trials_raw)}')

# Interpolar huecos cortos y filtrar trials con huecos interiores grandes
usable_trials = []
for t in trials_raw:
    t.X, _ = interpolate_short_gaps(t.X, max_gap=MAX_GAP)
    _, worst_interior = analyze_gaps(t.X)
    if worst_interior <= MAX_GAP:
        usable_trials.append(t)

print(f'Trials usables tras limpieza: {len(usable_trials)}')

## 2. Subsampling ×2: 100 fps → 50 fps

El paper §3.2 indica: *"measurements were subsampled by a factor 2 reducing the sampling rate from 100 frames/sec to 50 frames/sec."*
`process_trial()` aplica esto automáticamente. Aquí lo visualizamos.

In [ ]:
t = usable_trials[0]
xyz_full = masked_coords(t.X)
xyz_sub  = xyz_full[::SUBSAMPLE_FACTOR]

fig, ax = plt.subplots(figsize=(10, 3))
rca_idx = MARKER_IDX['RCA']
ax.plot(xyz_full[:, rca_idx, 1], label=f'100 fps ({len(xyz_full)} frames)', alpha=0.6)
ax.plot(np.arange(0, len(xyz_full), SUBSAMPLE_FACTOR),
        xyz_sub[:, rca_idx, 1],  label=f'50 fps  ({len(xyz_sub)} frames)', alpha=0.8)
ax.set_title('Marcador RCA (talón derecho) — posición vertical Y')
ax.set_xlabel('frame'); ax.set_ylabel('mm'); ax.legend(); plt.tight_layout()

## 3. Segmentación de pasos (Tasks 1+2)

Detectamos foot-strikes por mínimos locales en la posición vertical del talón.  
El trial se recorta para que empiece en el primer foot-strike y termine en el último.

In [ ]:
# Visualizar detección de pasos en un trial de ejemplo
t_demo = usable_trials[10]
xyz_demo = masked_coords(t_demo.X)[::SUBSAMPLE_FACTOR]

r_strikes, l_strikes = detect_steps(xyz_demo)
seg = segment_trial(xyz_demo)

print(f'Trial: {t_demo.trial_id} | Clase: Form {t_demo.label+1}')
print(f'Frames totales (50fps): {len(xyz_demo)}')
if seg:
    print(f'Foot-strikes derecho: {len(r_strikes)}')
    print(f'Foot-strikes izquierdo: {len(l_strikes)}')
    print(f'Pasos detectados: {seg["n_steps"]}')
    print(f'Período T: {seg["T_frames"]:.1f} frames = {seg["T_frames"]/FS*1000:.0f} ms')
    print(f'Frames válidos: {len(seg["xyz_cropped"])} '
          f'(de frame {seg["start_frame"]} a {seg["end_frame"]})')

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
rca_v = xyz_demo[:, MARKER_IDX['RCA'], 1]
lca_v = xyz_demo[:, MARKER_IDX['LCA'], 1]

axes[0].plot(rca_v, color='royalblue', label='RCA (talón derecho)')
axes[0].plot(r_strikes, rca_v[r_strikes], 'v', color='red', ms=8, label='foot-strike')
if seg:
    axes[0].axvspan(seg['start_frame'], seg['end_frame'], alpha=0.1, color='green',
                    label='región válida')
axes[0].set_ylabel('Posición Y (mm)'); axes[0].legend(fontsize=8)

axes[1].plot(lca_v, color='darkorange', label='LCA (talón izquierdo)')
axes[1].plot(l_strikes, lca_v[l_strikes], 'v', color='red', ms=8, label='foot-strike')
if seg:
    axes[1].axvspan(seg['start_frame'], seg['end_frame'], alpha=0.1, color='green')
axes[1].set_xlabel('Frame (50 fps)'); axes[1].set_ylabel('Posición Y (mm)')
axes[1].legend(fontsize=8)
plt.suptitle(f'Segmentación de pasos — Trial {t_demo.trial_id}'); plt.tight_layout()

In [ ]:
# Estadísticas de segmentación sobre todos los trials
step_counts = []
for t in usable_trials:
    xyz_s = masked_coords(t.X)[::SUBSAMPLE_FACTOR]
    seg_r = segment_trial(xyz_s)
    if seg_r:
        step_counts.append({'label': t.label, 'n_steps': seg_r['n_steps'],
                            'T_ms': seg_r['T_frames']/FS*1000})

df_steps = pd.DataFrame(step_counts)
print(f'Trials con pasos válidos: {len(df_steps)}/{len(usable_trials)}')
print(f'Trials descartados por pocos pasos: {len(usable_trials) - len(df_steps)}')
print(f'\nEstadísticas de N pasos por trial:')
print(df_steps['n_steps'].describe())
print(f'\nEstadísticas del período T (ms):')
print(df_steps['T_ms'].describe())

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
df_steps['n_steps'].hist(bins=20, ax=ax[0])
ax[0].set_title('Distribución de N pasos por trial'); ax[0].set_xlabel('N pasos')
df_steps['T_ms'].hist(bins=30, ax=ax[1])
ax[1].set_title('Distribución del período T (ms)'); ax[1].set_xlabel('T (ms)')
plt.tight_layout()

## 4. 81 ángulos proyectados (Task 3)

Proyectamos las 3 coordenadas en cada plano anatómico y calculamos los 27 ángulos  
de la Tabla 5 del paper. Resultado: **81 ángulos por frame**.

In [ ]:
# Calcular ángulos en el trial de demo
if seg:
    angles_demo = compute_projected_angles(seg['xyz_cropped'])
    print(f'Shape ángulos: {angles_demo.shape}  (frames, 81 ángulos)')
    print(f'NaN por columna (% frames inválidos): '
          f'{np.isnan(angles_demo).mean(axis=0).max():.1%} máx')

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    plane_names = [p for p, _ in PLANES]
    for k, (pname, ax) in enumerate(zip(plane_names, axes)):
        block = angles_demo[:, k*27:(k+1)*27]
        ax.plot(block, alpha=0.5, lw=0.8)
        ax.set_ylabel(f'Plano {pname}\n(grados)')
    axes[-1].set_xlabel('Frame (50 fps)')
    plt.suptitle(f'81 ángulos por frame — Trial {t_demo.trial_id} (Form {t_demo.label+1})')
    plt.tight_layout()

## 5. FFT → Features MLP (Task 4a)

Aplicamos FFT y extraemos los primeros 20 coeficientes armónicos del período de un paso.  
Vector final: **20 × 81 = 1620 features** por trial.

In [ ]:
if seg:
    mlp_demo = compute_mlp_features(angles_demo, n_steps=seg['n_steps'])
    print(f'Shape features MLP: {mlp_demo.shape}  (esperado: {N_COEFF*81})')

    # Visualizar el vector de features como heatmap
    feat_matrix = mlp_demo.reshape(N_COEFF, 81)   # (20 coefs, 81 ángulos)
    fig, ax = plt.subplots(figsize=(14, 4))
    im = ax.imshow(feat_matrix, aspect='auto', cmap='viridis')
    ax.set_xlabel('Ángulo (0-80)')
    ax.set_ylabel('Coeficiente armónico (k=1..20)')
    ax.set_title(f'Heatmap de features MLP — Trial {t_demo.trial_id} (N pasos={seg["n_steps"]})')
    plt.colorbar(im, ax=ax, label='Amplitud (normalizada)')
    plt.tight_layout()

    # FFT del ángulo 0 (primer ángulo del plano sagital) — visualización
    col0 = angles_demo[:, 0].copy()
    col0[np.isnan(col0)] = np.nanmean(col0)
    fft_vals = np.abs(np.fft.rfft(col0))
    N = seg['n_steps']
    harmonic_idx = np.arange(1, N_COEFF+1) * N

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.semilogy(fft_vals, color='steelblue', lw=0.8, label='Espectro FFT')
    ax.axvline(N, color='red', ls='--', label=f'Fundamental (k=N={N})')
    ax.plot(harmonic_idx[harmonic_idx < len(fft_vals)],
            fft_vals[harmonic_idx[harmonic_idx < len(fft_vals)]],
            'ro', ms=6, label='Armónicos extraídos')
    ax.set_xlabel('Índice FFT'); ax.set_ylabel('Amplitud (log)')
    ax.set_title('Espectro FFT — ángulo 0 (sagital, triplete 0)')
    ax.legend(); plt.tight_layout()

## 6. Secuencias para LSTM (Task 4b)

Dividimos la secuencia de ángulos en ventanas de **75 frames**, desplazadas  
en **15 frames**, con un máximo de **45 secuencias** por trial.

In [ ]:
if seg:
    lstm_demo = compute_lstm_sequences(angles_demo)
    print(f'Shape secuencias LSTM: {lstm_demo.shape}  '
          f'(esperado: (n_seq ≤ {MAX_SEQ}, {SEQ_LEN}, 81))')

    # Visualizar la primera y última ventana
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax_i, idx, title in zip(axes, [0, -1], ['Primera ventana', 'Última ventana']):
        ax_i.imshow(lstm_demo[idx].T, aspect='auto', cmap='RdBu_r',
                    vmin=0, vmax=180)
        ax_i.set_xlabel('Frame (dentro de la ventana)')
        ax_i.set_ylabel('Ángulo (0-80)')
        ax_i.set_title(title)
    plt.suptitle(f'Heatmap de secuencias LSTM — Trial {t_demo.trial_id}')
    plt.tight_layout()

## 7. Construcción del dataset completo

Procesamos todos los trials y verificamos que el conteo se acerque a la Tabla 3 del paper:

| Form | Train | Test | Total |
|------|-------|------|-------|
| 1    | 47    | 16   | 63    |
| 2    | 183   | 83   | 266   |
| 3    | 174   | 49   | 223   |
| 4    | 372   | 114  | 486   |
| **Total** | **776** | **262** | **1038** |

In [ ]:
print('Construyendo dataset completo (puede tardar unos minutos)...')
dataset = build_dataset(usable_trials, verbose=True)

mlp  = dataset['mlp']
lstm = dataset['lstm']

print(f'\n=== Dataset MLP ===')
print(f'X shape:  {mlp["X"].shape}   (trials × 1620 features)')
print(f'y shape:  {mlp["y"].shape}')
print('Distribución por clase:')
labels, counts = np.unique(mlp['y'], return_counts=True)
for l, c in zip(labels, counts):
    print(f'  Form {l+1}: {c} trials')
print(f'  TOTAL:  {mlp["X"].shape[0]} trials')

print(f'\n=== Dataset LSTM ===')
print(f'X shape:  {lstm["X"].shape}   (secuencias × 75 frames × 81 ángulos)')
labels_l, counts_l = np.unique(lstm['y'], return_counts=True)
for l, c in zip(labels_l, counts_l):
    print(f'  Form {l+1}: {c} secuencias')
print(f'  TOTAL:  {lstm["X"].shape[0]} secuencias')

In [ ]:
# Comparar con la Tabla 3 del paper
paper_mlp = {0: 63, 1: 266, 2: 223, 3: 486}   # total trials por clase

print('Comparación con Tabla 3 del paper (total trials, sin split aún):')
print(f'{"Clase":<8} {"Nuestros":>10} {"Paper":>8} {"Diferencia":>12}')
print('-' * 42)
for l, c in zip(labels, counts):
    paper_c = paper_mlp.get(l, '?')
    diff = c - paper_c if isinstance(paper_c, int) else '?'
    print(f'Form {l+1}   {c:>10} {paper_c:>8} {diff:>12}')
print(f'{"TOTAL":<8} {mlp["X"].shape[0]:>10} {1038:>8} {mlp["X"].shape[0]-1038:>12}')

In [ ]:
# Visualización resumen del dataset
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels_str = [f'Form {l+1}' for l in labels]
axes[0].bar(labels_str, counts, color=['#e74c3c','#2ecc71','#3498db','#f39c12'])
axes[0].set_title('Trials por clase — MLP dataset'); axes[0].set_ylabel('Número de trials')

labels_sl = [f'Form {l+1}' for l in labels_l]
axes[1].bar(labels_sl, counts_l, color=['#e74c3c','#2ecc71','#3498db','#f39c12'])
axes[1].set_title('Secuencias por clase — LSTM dataset'); axes[1].set_ylabel('Número de secuencias')
plt.tight_layout()

## 8. Guardar datasets

Guardamos los arrays en `data/features/` para usarlos en los notebooks de modelos.

In [ ]:
# Guardar dataset MLP
mlp_path = os.path.join(OUT_DIR, 'mlp_features.npz')
np.savez(
    mlp_path,
    X       = mlp['X'],
    y       = mlp['y'],
    subject = mlp['subject'],
)
print(f'Guardado: {mlp_path}')

# Guardar dataset LSTM
lstm_path = os.path.join(OUT_DIR, 'lstm_sequences.npz')
np.savez(
    lstm_path,
    X       = lstm['X'],
    y       = lstm['y'],
    subject = lstm['subject'],
)
print(f'Guardado: {lstm_path}')

print('\n✓ Etapa B completada. Datasets listos para los notebooks de modelos.')